# Zuber: análisis de viajes en taxi en Chicago

**Autor:** Juan Pablo Quintero

## Descripción del proyecto

Zuber es una nueva empresa de viajes compartidos que se lanza en Chicago. Como analista, mi tarea es encontrar patrones en los datos disponibles para entender las preferencias de los pasajeros y el impacto de factores externos en los viajes.

Para eso voy a:

1. Obtener por web scraping los datos del clima de Chicago en noviembre de 2017.
2. Analizar con SQL los datos de las empresas de taxis de la competencia.
3. Probar la hipótesis de que el clima afecta la duración de los viajes desde el Loop hasta el Aeropuerto Internacional O'Hare.


## Descripción de los datos

La base de datos contiene información sobre viajes en taxi en Chicago:

| Tabla             | Contenido            | Campos principales                                                                                                           |
| ----------------- | -------------------- | ---------------------------------------------------------------------------------------------------------------------------- |
| `neighborhoods`   | Barrios de la ciudad | `neighborhood_id`, `name`                                                                                                    |
| `cabs`            | Taxis                | `cab_id`, `vehicle_id`, `company_name`                                                                                       |
| `trips`           | Viajes               | `trip_id`, `cab_id`, `start_ts`, `end_ts`, `duration_seconds`, `distance_miles`, `pickup_location_id`, `dropoff_location_id` |
| `weather_records` | Clima por hora       | `record_id`, `ts`, `temperature`, `description`                                                                              |

**Nota:** `trips` y `weather_records` no tienen una relación directa. Se unen por la hora de inicio del viaje (`trips.start_ts`) y la hora del registro meteorológico (`weather_records.ts`).


## Importar librerías


In [1]:
# Web scraping
import requests
from bs4 import BeautifulSoup

# Manipulación de datos
import pandas as pd

---

## Paso 1. Web scraping del clima

**Objetivo:** extraer de una página web la tabla con el clima por hora de Chicago en noviembre de 2017.

**Para qué sirve:** estos registros son la base para clasificar cada hora como _Good_ o _Bad_ según el clima (Paso 3) y probar la hipótesis sobre la duración de los viajes (Paso 5).


In [2]:
# URL de la página con los registros del clima de Chicago (noviembre de 2017)
URL = 'https://practicum-content.s3.us-west-1.amazonaws.com/data-analyst-eng/moved_chicago_weather_2017.html'

# Petición GET: descarga la página. La respuesta queda en `req`
req = requests.get(URL)

# Parseo: convierte el HTML (req.text) en un objeto navegable usando el parser lxml
soup = BeautifulSoup(req.text, 'lxml')

In [3]:
# Ubicar la tabla por su id (find devuelve un solo elemento, no una lista)
table = soup.find('table', attrs={'id': 'weather_records'})

# Encabezados: texto de cada etiqueta <th>
heading_table = [row.text for row in table.find_all('th')]

# Filas de datos: recorre cada <tr> y salta la fila de encabezados (la que tiene <th>).
# De las demás toma el texto de cada celda <td>
content = []
for row in table.find_all('tr'):
    if not row.find_all('th'):
        content.append([element.text for element in row.find_all('td')])

In [4]:
# Construir el DataFrame con las filas extraídas y los encabezados como nombres de columna
weather_records = pd.DataFrame(content, columns=heading_table)

# Imprimir el DataFrame completo
print(weather_records)

           Date and time Temperature       Description
0    2017-11-01 00:00:00     276.150     broken clouds
1    2017-11-01 01:00:00     275.700  scattered clouds
2    2017-11-01 02:00:00     275.610   overcast clouds
3    2017-11-01 03:00:00     275.350     broken clouds
4    2017-11-01 04:00:00     275.240     broken clouds
..                   ...         ...               ...
692  2017-11-29 20:00:00     281.340        few clouds
693  2017-11-29 21:00:00     281.690      sky is clear
694  2017-11-29 22:00:00     281.070        few clouds
695  2017-11-29 23:00:00     280.060      sky is clear
696  2017-11-30 00:00:00     278.460      sky is clear

[697 rows x 3 columns]


### Revisión del DataFrame obtenido


In [ ]:
# Dimensiones: (filas, columnas)
print(weather_records.shape)

In [ ]:
# Tipos de datos y valores no nulos por columna
weather_records.info()

In [ ]:
# Primeras y últimas filas, para confirmar que el rango de fechas es completo
display(weather_records.head())
display(weather_records.tail())

In [ ]:
# Guardar una copia local del clima obtenido, para que el proyecto
# sea reproducible aunque la página deje de estar disponible
weather_records.to_csv('../data/chicago_weather_nov_2017.csv', index=False)

### Observaciones del Paso 1

> _Completar después de ejecutar:_
>
> - Número de registros obtenidos y período que cubren.
> - Columnas y tipo de dato de cada una. El scraping devuelve todo como texto (`object`): `Date and time` debería pasar a `datetime` y `Temperature` a numérico si se usaran en un análisis.
> - ¿Hay valores nulos o filas vacías?


---

## Paso 2. Análisis exploratorio de datos (SQL)

Las consultas de este paso se ejecutaron en la base de datos PostgreSQL de la plataforma de TripleTen. Cada consulta también está guardada como archivo en la carpeta `sql/`.

**Técnicas usadas:** `INNER JOIN` entre `trips` y `cabs` por `cab_id`, filtros de fecha con `CAST(... AS DATE)`, agregación con `COUNT` y `GROUP BY`, y filtrado de texto con `LIKE`.

### Ejercicio 1. Viajes por empresa el 15 y 16 de noviembre de 2017

**Pregunta:** ¿cuántos viajes hizo cada empresa de taxis entre el 15 y el 16 de noviembre de 2017?

```sql
-- Paso 2 · Ejercicio 1
-- Cantidad de viajes en taxi por empresa entre el 15 y el 16 de noviembre de 2017,
-- ordenada de mayor a menor (resultado exportado como project_sql_result_01.csv).

SELECT
    c.company_name,
    COUNT(t.trip_id) AS trips_amount
FROM
    trips t
    INNER JOIN cabs c ON t.cab_id = c.cab_id
WHERE
    CAST(t.start_ts AS DATE) BETWEEN '2017-11-15' AND '2017-11-16'
GROUP BY
    c.company_name
ORDER BY
    trips_amount DESC;
```

**Nota técnica:** `start_ts` es un *timestamp*. Comparar directamente con `BETWEEN '2017-11-15' AND '2017-11-16'` dejaría fuera casi todo el día 16 (se interpretaría como `2017-11-16 00:00:00`). Por eso se convierte primero a fecha con `CAST(... AS DATE)`.

**Resultado (primeras 5 de 64 empresas):**

| company_name | trips_amount |
|---|---:|
| Flash Cab | 19,558 |
| Taxi Affiliation Services | 11,422 |
| Medallion Leasin | 10,367 |
| Yellow Cab | 9,888 |
| Taxi Affiliation Service Yellow | 9,299 |

**Observaciones:**
- **Mercado concentrado:** Flash Cab tiene casi el doble de viajes que la segunda empresa.
- **Cola larga:** cerca de la mitad de las 64 empresas tienen menos de 100 viajes. Muchas parecen conductores independientes registrados con un código delante del nombre (por ejemplo, `6742 - 83735 Tasha ride inc`).
- **Calidad de los nombres:** hay nombres que podrían ser variantes o truncamientos de la misma empresa (`Medallion Leasin` / `Chicago Medallion Leasing INC`, `Checker Taxi` / `Checker Taxi Affiliation`, `American United` / `American United Taxi Affiliation`). No se unificaron porque no hay forma de confirmar que sean la misma empresa.

### Ejercicio 2. Empresas "Yellow" y "Blue" del 1 al 7 de noviembre de 2017

**Pregunta:** ¿cuántos viajes hizo cada empresa cuyo nombre contiene "Yellow" o "Blue" entre el 1 y el 7 de noviembre de 2017?

```sql
-- Paso 2 · Ejercicio 2
-- Cantidad de viajes por empresa cuyo nombre contiene "Yellow" o "Blue",
-- entre el 1 y el 7 de noviembre de 2017.

SELECT
    c.company_name AS company_name,
    COUNT(t.trip_id) AS trips_amount
FROM
    trips t
    INNER JOIN cabs c ON t.cab_id = c.cab_id
WHERE
    CAST(t.start_ts AS DATE) BETWEEN '2017-11-01' AND '2017-11-07'
    -- Los paréntesis son necesarios: AND se evalúa antes que OR
    AND (c.company_name LIKE '%Yellow%' OR c.company_name LIKE '%Blue%')
GROUP BY
    c.company_name
ORDER BY
    trips_amount DESC;
```

**Resultado:**

| company_name | trips_amount |
|---|---:|
| Yellow Cab | 33,668 |
| Taxi Affiliation Service Yellow | 29,213 |
| Blue Ribbon Taxi Association Inc. | 17,675 |
| Blue Diamond | 6,764 |

**Observaciones:**
- Las dos empresas "Yellow" están casi parejas, y cada una supera con amplitud a las "Blue".
- Los líderes del mercado (como Flash Cab en el ejercicio 1) no tienen nombres de "marca de color", así que este filtro no captura a los competidores más fuertes.

### Ejercicio 3. Flash Cab, Taxi Affiliation Services y "Other" en noviembre de 2017

> _Pendiente_

---

## Paso 3. Consultas para probar la hipótesis (SQL)

> _Pendiente_


---

## Paso 4. Análisis exploratorio de datos (Python)

> _Pendiente_


---

## Paso 5. Prueba de hipótesis

> _Pendiente_


---

## Conclusión general

> _Pendiente_
